In [10]:
import pandas as pd
import sys

sys.path.append('../python_scripts')





In [11]:
run_name = '0d_quality_fixed'

df_median = pd.read_parquet("../sim_results/{}/median_objectives".format(run_name))
df_median.to_parquet('../sim_results/{}/median_objectives.parquet'.format(run_name))

In [12]:
df = pd.read_parquet('../sim_results/{}/median_objectives.parquet'.format(run_name)).reset_index()

In [13]:
df_train_regret = df.loc[(df['mode'] == 'train') & (df.objective == 'mean_regret')]
df_val_regret = df.loc[(df['mode'] == 'val') & (df.objective == 'mean_regret')]
df_val = df.loc[(df['mode'] == 'val') & (df.objective == 'mean')]
df_train = df.loc[(df['mode'] == 'train') & (df.objective == 'mean')]

s_train = df_train[['gen', 'value']].groupby('gen').agg('mean').iloc[:,0].sort_index()
s_train_regret = df_train_regret[['gen', 'value']].groupby('gen').agg('mean').iloc[:,0].sort_index()
s_val_regret = df_val_regret[['gen', 'value']].groupby('gen').agg('mean').iloc[:, 0].sort_index()
s_val = df_val[['gen', 'value']].groupby('gen').agg('mean').iloc[:, 0].sort_index()


In [14]:
import plotly.graph_objects as go 

fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.add_trace(go.Scatter(x = s_train.index, y = s_train.values, name = 'train'))
fig.show()

In [15]:
from utils import get_pareto_layers

df_pareto = pd.DataFrame()
df_1  = pd.DataFrame()
for gen in df['gen'].drop_duplicates():
    df_gen = df.loc[df['gen'] == gen]
    df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
    df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=1)
    pareto_indices = df_layers.loc[df_layers.layer==0].index
    df_add = df_pivot.loc[pareto_indices]
    df_add['gen'] = gen
    argmin = int(df_add[('train', 'mean_regret')].argmin())
    df_add_1 = pd.DataFrame(df_add.iloc[argmin]).transpose() 
    df_1 = pd.concat([df_1, df_add_1])
    df_pareto = pd.concat([df_pareto, df_add])
    
df_pareto.to_parquet('../sim_results/{}/train_pareto_by_gen.parquet'.format(run_name))


In [9]:
#ALTERNATIVE
from utils import get_pareto_layers

df_pareto = pd.DataFrame()
df_1  = pd.DataFrame()
for gen in sorted(list(df['gen'].drop_duplicates())):
    df_gen = df.loc[df['gen'] == gen]
    df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
    #df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=1)
    #pareto_indices = df_layers.loc[df_layers.layer==0].index
    df_add = df_pivot
    df_add['gen'] = gen
    argmin = int(df_add[('train', 'mean_regret')].argmin())
    df_add_1 = pd.DataFrame(df_add.iloc[argmin]).transpose() 
    df_1 = pd.concat([df_1, df_add_1])
    df_pareto = pd.concat([df_pareto, df_add])
    if sim_id in df_pivot.index:
        print('yes', gen, sim_id in df_gen.sim_id, )
        zzz=1
    
    
df_pareto.to_parquet('../sim_results/{}/train_pareto_by_gen.parquet'.format(run_name))

KeyError: 'gen'

In [137]:
df_1.gen.dtype

dtype('float64')

In [ ]:
s_train_regret = (df_pareto[[('train', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val_regret = (df_pareto[[('val', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val= (df_pareto[[('val', 'mean'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.show()

: 

In [8]:
import plotly.graph_objects as go 
s_train_regret = (df_1[[('train', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val_regret = (df_1[[('val', 'mean_regret'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
s_val = (df_1[[('val', 'mean'), ('gen','')]]).groupby('gen').agg('mean').iloc[:,0]
fig = go.Figure()
fig.add_trace(go.Scatter(x = s_train_regret.index.values, y = s_train_regret.values, name = 'train_regret'))
fig.add_trace(go.Scatter(x = s_val_regret.index, y = s_val_regret.values, name = 'val_regret'))
fig.add_trace(go.Scatter(x = s_val.index, y = s_val.values, name = 'val'))
fig.show()

In [2]:
import xarray as xr
import pandas as pd
s_voo = xr.open_dataset('../simulation_data/momentum.nc').to_array()[0].sel(symbol = 'VOO', band = 'price_end').to_pandas()
fold_index = 4
df_folds = pd.read_parquet('../strategy/folds.parquet')
df_fold = df_folds.loc[df_folds.fold_index == fold_index]

In [5]:
import functools
from objective_functions import mean_annualized_return, WeightedRegimeApplyer, weighted_mean


start_date = min(df_fold.start_date)
end_date = max(df_fold.end_date)
agg_func = functools.partial(mean_annualized_return, start_date, end_date)

voo_func = WeightedRegimeApplyer(df_fold, agg_func, weighted_mean)
voo_mean = voo_func(s_voo)
voo_mean


np.float64(0.15859726102530505)

In [70]:
import functools
from objective_functions import mean_annualized_return, WeightedRegimeApplyer, weighted_mean
df_folds = pd.read_csv('../strategy/folds_1.csv')
for col in ['start_date', 'end_date']:
    df_folds[col] = pd.to_datetime(df_folds[col])
df_voo = pd.read_csv('../simulation_data/voo_backcasted_28days.csv')
s_voo = pd.Series(df_voo.values[:,-1], index = pd.to_datetime(df_voo.iloc[:,0]))
df_out = pd.DataFrame()
for row in range(df_folds.shape[0]):
    df_fold = df_folds.iloc[row].copy()
    start_date, end_date = df_fold[['start_date', 'end_date']]
    

    
    voo_mean = mean_annualized_return(start_date, end_date,s_voo)
    df_fold['voo_return'] = voo_mean
    df_out = pd.concat([df_out, pd.DataFrame(df_fold.copy()).transpose()])
    print(start_date, end_date, voo_mean)
df_out = df_out.rename(columns = {'index': 'description'}).set_index('description')
print(df_out)
        

2008-01-01 00:00:00 2008-08-31 00:00:00 -0.08560702196953351
2008-09-01 00:00:00 2009-03-31 00:00:00 -0.6255475194744984
2009-04-01 00:00:00 2009-12-31 00:00:00 0.4579419345017437
2010-01-01 00:00:00 2011-04-30 00:00:00 0.13197869183563848
2011-05-01 00:00:00 2011-11-30 00:00:00 -0.18230933045622366
2011-12-01 00:00:00 2013-04-30 00:00:00 0.248253698361115
2013-05-01 00:00:00 2013-12-31 00:00:00 0.20293260341316
2014-01-01 00:00:00 2015-07-31 00:00:00 0.10908982197281247
2015-08-01 00:00:00 2016-02-29 00:00:00 -0.022362341330139923
2016-03-01 00:00:00 2017-12-31 00:00:00 0.19914092503598924
2018-01-01 00:00:00 2018-03-31 00:00:00 -0.039952295425778384
2018-04-01 00:00:00 2018-09-30 00:00:00 0.30276171252467443
2018-10-01 00:00:00 2018-12-31 00:00:00 -0.2247335172284438
2019-01-01 00:00:00 2020-01-31 00:00:00 0.2997640483772288
2020-02-01 00:00:00 2020-04-30 00:00:00 -0.3967135888024208
2020-05-01 00:00:00 2021-12-31 00:00:00 0.3420393562675983
2022-01-01 00:00:00 2022-09-30 00:00:00 -0

In [78]:

mid_point = df_out.start_date.min() + pd.Timedelta(days = .75 * (pd.to_datetime('Jan 1, 2025') - df_out.start_date.min()).days)
mid_point

Timestamp('2020-10-01 12:00:00')

In [95]:
df_out.to_csv('../strategy/folds_2.csv')
for col in ['start_date', 'end_date']:
    df_out[col] = pd.to_datetime(df_out[col])

In [97]:
df2 = df_out.copy()

df2['fold_index'] = 1
df2.loc[df2.start_date <= pd.to_datetime('2020-05-01'),'fold_index'] = 0
df2.loc[df2.start_date >= pd.to_datetime('2024-09-01'), 'fold_index'] =2
df2.to_parquet('../strategy/folds_1.parquet')

In [94]:
pd.to_datetime(df_out.end_date)

description
Early GFC Recession (Pre-Lehman)                        2008-08-31
GFC Panic Apex (Lehman to Market Bottom)                2009-03-31
Transition: Liquidity-Driven Early Recovery             2009-12-31
Mid-Cycle Expansion & QE1/QE2                           2011-04-30
Transition: Euro Sovereign Debt & US Downgrade          2011-11-30
Draghi "Whatever It Takes" Stabilization                2013-04-30
Transition: Taper Tantrum Rate Shock                    2013-12-31
US Decoupling & Strong Dollar Inflow                    2015-07-31
Transition: China Devaluation & Oil Crash               2016-02-29
Post-Election Reflation & Low-Vol Goldilocks            2017-12-31
Transition: Volmageddon Short-Volatility Squeeze        2018-03-31
Trade War Escalation & Synchronized Slowdown            2018-09-30
Transition: Fed "Long Way From Neutral" Hawk Crash      2018-12-31
Fed Dovish Pivot & Insurance Rate Cuts                  2020-01-31
Transition: COVID-19 Flash Liquidation Shock      

In [16]:
sim_id = df_1.index[-1]
s3_path = "s3://jdinvestment/{}/portfolio_values/sim_{}.parquet".format(run_name, sim_id)
s_values = pd.read_parquet(s3_path).transpose().iloc[:-2,0]
s_values.index = pd.to_datetime(s_values.index.str.replace('_',' '))
opt_mean = voo_func(s_values)
sim_id, opt_mean, sim_id in df_1.index



('15c81be3b60f', np.float64(-0.03826888266447126), True)

In [155]:
import numpy as np
df_pop = pd.read_parquet('../sim_results/{}/populations/gen_149.parquet'.format(run_name))
sim_index = np.where(df_pop.index == sim_id)[0][0]
sim_index

np.int64(176)

In [142]:
sim_id in df_1.index




True

In [54]:
df_holdings = pd.read_parquet('s3://jdinvestment/2d_test_fold_2/holdings/sim_a99516312234.parquet')

In [102]:
df19 = df_holdings.loc[df_holdings.date.dt.year == 2019].set_index('date')
df19.index = pd.to_datetime(df19.index)
df19.drop('sim_id', axis = 1, inplace = True)
df19 = pd.DataFrame(df19.values/df19.values.sum(1).reshape(df19.shape[0],1), index = df19.index, columns = df19.columns)
df19 = df19.loc[:, df19.max(0) > 0]
df19.mean(0).sort_index(ascending = False)



symbol
XOM     0.007691
XLG     0.003846
WPC     0.003844
WM      0.007689
VRTX    0.003846
VRT     0.008142
TSLA    0.003846
TMUS    0.026552
STAG    0.007692
SCHG    0.042308
RSG     0.008693
REXR    0.011538
PLD     0.015381
PGR     0.015384
ORLY    0.042306
O       0.007691
NNN     0.003846
MSFT    0.011536
MPWR    0.013926
MPC     0.007692
LLY     0.003846
KLAC    0.003846
KKR     0.003848
GILD    0.003844
GIC     0.445235
FR      0.000002
FICO    0.010030
EXEL    0.003846
EGP     0.011533
DECK    0.049999
DBMF    0.003847
CWST    0.046150
CVX     0.011535
CASY    0.000201
BX      0.030769
BTAL    0.011539
BKNG    0.026917
ARES    0.042306
APO     0.019229
APH     0.010338
ADC     0.007692
dtype: float64

In [114]:
df_pop = pd.read_parquet('../sim_results/2d_test_fold_2/populations/gen_149.parquet')
df_pop.shape

(215, 21)

In [115]:
sim_id

'a99516312234'